# 01 · Data limpieza & ingesta (Bronze)

**Configuración base:**
- Catálogo: `main`
- Schemas: `loterias_raw`, `loterias_bronze`, `loterias_silver`, `loterias_features`
- Volume crudos: `/Volumes/main/loterias_raw/raw_apuestas/apuestas_partitioned/`

In [0]:
CATALOG = "main"
SCHEMA_BRONZE = "loterias_bronze"
DATA_PATH = "/Volumes/main/loterias_raw/raw_apuestas/apuestas_partitioned/"
TABLE_BRONZE = f"{CATALOG}.{SCHEMA_BRONZE}.apuestas_bronze"

from pyspark.sql import functions as F
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_BRONZE}")
df = (spark.read.option("header", True).csv(DATA_PATH))
required = ["tx_id","user_id","canal","monto","min_antes_cierre","ip","fecha","es_fraude"]
missing = [c for c in required if c not in df.columns]
assert not missing, f"Faltan columnas: {missing}"
(df.write.format("delta").mode("overwrite").partitionBy("fecha").saveAsTable(TABLE_BRONZE))
display(spark.table(TABLE_BRONZE).limit(10))